# FactFlip

This is an example notebook showing the use of FactFlip on the FM2 dataset, as described in the "Logit-Based Universal Trigger Search for Attacking Claim Verification Models" paper.
In particular, in this notebook we show how to rank the triggers, integrate them inside the claims based on the FactFlip-RAW setting, and test the attack success rate on the claim verification model.

This notebook does not depend on external Python code. However, to run it you need have the `data/` folder in the same directory (with the `fm2/` and `antonym/` subfolders inside it), and the finetuned roberta-base model inside `models/roberta-base/seed_1/fm2/fm2_model.pt`.

The code inside this notebook can handle inference with the `Qwen2.5-14B-Instruct` model, and also the generation of claims with OpenAI, with some minor code tweaks (e.g. changes in the `config` dictionary, or changes to the `cv_attack` function respectively). However, it does not handle other datasets other than FM2, neither does it handle the NEI class. Moreover, it does not handle the injection of multiple triggers, nor the similarity setting (FactFlip-SIM) and the dev tuning setting (FactFlip-DS). For more general use cases, please refer to the other python scripts in this repository.

The code is structured as follows:
1. **Util functions**, containing the code for general functions, e.g. setting random seeds, computing metrics, etc.;
2. **Data Processor**, containing the code for data processing (loading of the datasets) and tokenization;
3. **Model Definitions**, containing the code for the model definition (Roberta-based claim verification model, and Qwen-based generative model), as well as the OpenAI API wrapper;
4. **Trainer definition**, containing the code for training and evaluation of the claim verification models, as well as the code for ranking the triggers;
5. **Main script**, containing
    - the code to extract the triggers;
    - the code to keep only the dataset instances that the model correctly predicts;
    - the code to inject the triggers inside the claims based on the FactFlip-RAW setting;
    - the code to evaluate the attack success rate of the triggers on the claim verification model.

## Util functions

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import torch
import random

import numpy as np

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from functools import lru_cache

@lru_cache()
def get_device():
    device = torch.device("cpu")
    if torch.cuda.is_available():
        print("Training on GPU")
        device = torch.device("cuda:0")

    return device

def set_random_seeds(seed):
    """
    set random seed
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

def output_metrics(labels, preds):
    """

    :param labels: ground truth labels
    :param preds: prediction labels
    :return: accuracy, precision, recall, f1
    """
    accuracy = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, average="macro")
    recall = recall_score(labels, preds, average="macro")
    f1 = f1_score(labels, preds, average="macro")

    print("{:15}{:<.6f}".format('accuracy:', accuracy))
    print("{:15}{:<.6f}".format('precision:', precision))
    print("{:15}{:<.6f}".format('recall:', recall))
    print("{:15}{:<.6f}".format('f1:', f1))

    return accuracy, precision, recall, f1


## Data Processor

In [3]:
import pandas as pd
import json
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer
from tqdm import tqdm

In [4]:
class dataset(Dataset):
    def __init__(self, examples):
        super(dataset, self).__init__()
        self.examples = examples

    def __getitem__(self, idx):
        return self.examples[idx]

    def __len__(self):
        return len(self.examples)


def collate_fn(examples):
    claim, evidence, ids_sent1, segs_sent1, att_mask_sent1, labels = map(list, zip(*examples))

    ids_sent1 = torch.tensor(ids_sent1, dtype=torch.long)
    segs_sent1 = torch.tensor(segs_sent1, dtype=torch.long)
    att_mask_sent1 = torch.tensor(att_mask_sent1, dtype=torch.long)
    labels = torch.tensor(labels, dtype=torch.long)

    return claim, evidence, ids_sent1, segs_sent1, att_mask_sent1, labels

def collate_fn_antonym(examples):
    try:
        sent1, sent2, ids_sent1, segs_sent1, att_mask_sent1, ids_sent2, segs_sent2, att_mask_sent2, label = map(list, zip(*examples))
    except:
        sent1, sent2, ids_sent1, segs_sent1, att_mask_sent1, ids_sent2, segs_sent2, att_mask_sent2 = map(list, zip(*examples))
        label = None

    ids_sent1 = torch.tensor(ids_sent1, dtype=torch.long)
    segs_sent1 = torch.tensor(segs_sent1, dtype=torch.long)
    att_mask_sent1 = torch.tensor(att_mask_sent1, dtype=torch.long)
    ids_sent2 = torch.tensor(ids_sent2, dtype=torch.long)
    segs_sent2 = torch.tensor(segs_sent2, dtype=torch.long)
    att_mask_sent2 = torch.tensor(att_mask_sent2, dtype=torch.long)
    if label is not None:
        label = torch.tensor(label, dtype=torch.long)
        return sent1, sent2, ids_sent1, segs_sent1, att_mask_sent1, ids_sent2, segs_sent2, att_mask_sent2, label

    return sent1, sent2, ids_sent1, segs_sent1, att_mask_sent1, ids_sent2, segs_sent2, att_mask_sent2

class DataProcessor:

    def __init__(self,config):
        self.config = config

        if self.config["backbone"] is None:
            self.tokenizer = AutoTokenizer.from_pretrained(self.config["model_name"])
        else:
            self.tokenizer = AutoTokenizer.from_pretrained(self.config["backbone"])

        self.max_sent_len = config["max_sent_len"]
        self.prompt = """You are a fact checking system. You must indicate whether the claim is supported, refuted or "not enough information" based on the given evidence.\nAfter "Answer: ", write exclusively "support", "refute" or "not enough information". Do not write anything else.\n\n{input}\nAnswer:"""
        self.is_generative = ("llama" in self.config["model_name"].lower()
                             or "qwen" in self.config["model_name"].lower()
                             or "gpt"  in self.config["model_name"].lower())

    def __str__(self,):
        pattern = """General data processor: \n\n Tokenizer: {}\n\nMax sentence length: {}""".format(self.config["model_name"], self.max_sent_len)
        return pattern

    def add_word(self, claim, label, word=None, to_class=None):
        if word is None or to_class is None:
            return claim, label
        claim = f"{word}. {claim}"

        to_class = to_class.lower().strip()
        num_classes = len(label)
        if to_class == "support":
            if num_classes == 2:
                label = [1,0]
            else:
                label = [1,0,0]
        elif to_class == "refute":
            if num_classes == 2:
                label = [0,1]
            else:
                label = [0,1,0]
        elif to_class == "nei":
            if num_classes == 2:
                raise ValueError("The model cannot predict the NEI class")
            label = [0,0,1]
        else:
            raise ValueError(f"Unknown target class provided: {to_class}")

        return claim, label

    def _get_examples_causal_lm(self, claim, evidence):
        count_truncated_samples = 0

        text = f"Claim: {claim}\nEvidence: {evidence.strip()}"
        prompt = self.prompt.format(input=text)

        ids_sent1 = self.tokenizer.encode(prompt)
        segs_sent1 = [0] * len(ids_sent1)

        pad_id = self.tokenizer.encode(self.tokenizer.pad_token, add_special_tokens=False)[0]

        if len(ids_sent1) < self.max_sent_len:
            res = self.max_sent_len - len(ids_sent1)
            att_mask_sent1 = [0] * res + [1] * len(ids_sent1) # left padding for causal lm
            ids_sent1 = [pad_id] * res + ids_sent1
            segs_sent1 += [0] * res
        else:
            ids_sent1 = ids_sent1[:self.max_sent_len]
            segs_sent1 = segs_sent1[:self.max_sent_len]
            att_mask_sent1 = [1] * self.max_sent_len
            count_truncated_samples += 1

        return prompt, ids_sent1, segs_sent1, att_mask_sent1, count_truncated_samples

    def _get_examples(self, dataset, dataset_type="train", add_space=False):
        examples = []
        count_truncated_samples = 0

        if self.tokenizer.pad_token is None:
            # safe fallback if the tokenizer does not have a pad token
            self.tokenizer.pad_token = self.tokenizer.eos_token
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id

        for i,row in enumerate(tqdm(dataset, desc="tokenizing...")):
            id, claim, evidence, label = row
            if add_space:
                claim = ". "+claim

            """
            for the first sentence
            """
            if len(evidence.strip()) == 0:
                evidence = "no evidence"

            if not self.is_generative:
                claim_length = len(self.tokenizer.encode(claim))
                evidence_length = len(self.tokenizer.encode(evidence))

                ids_sent1 = self.tokenizer.encode(claim, evidence)
                segs_sent1 = [0] * claim_length + [1] * evidence_length

                assert len(ids_sent1) == len(segs_sent1)

                pad_id = self.tokenizer.encode(self.tokenizer.pad_token, add_special_tokens=False)[0]

                if len(ids_sent1) < self.max_sent_len:
                    res = self.max_sent_len - len(ids_sent1)
                    att_mask_sent1 = [1] * len(ids_sent1) + [0] * res
                    ids_sent1 += [pad_id] * res
                    segs_sent1 += [0] * res
                else:
                    ids_sent1 = ids_sent1[:self.max_sent_len]
                    segs_sent1 = segs_sent1[:self.max_sent_len]
                    att_mask_sent1 = [1] * self.max_sent_len
                    count_truncated_samples += 1
            else:
                prompt_text, ids_sent1, segs_sent1, att_mask_sent1, truncated_count = self._get_examples_causal_lm(claim, evidence)
                count_truncated_samples += truncated_count

            example = [claim, evidence, ids_sent1, segs_sent1, att_mask_sent1, label]
            examples.append(example)

        print(f"finished preprocessing examples in {dataset_type}: {count_truncated_samples} samples truncated out of {len(dataset)}")

        return examples

class AntonymsProcessor(DataProcessor):

    def __init__(self, config):
        super(AntonymsProcessor, self).__init__(config)

    def _get_examples(self, dataset, dataset_type="train", add_space=False, template=0):
        examples = []
        count_truncated_samples = 0
        if self.tokenizer.pad_token is not None:
            pad_id = self.tokenizer.encode(self.tokenizer.pad_token, add_special_tokens=False)[0]
        else:
            pad_id = self.tokenizer.encode(self.tokenizer.eos_token, add_special_tokens=False)[0]

        for row in tqdm(dataset, desc="tokenizing..."):
            if len(row) == 4:
                id, sentence1, sentence2, label = row
            else:
                id, sentence1, sentence2 = row

            if template == 1:
                sentence1 = f"""The statement "{sentence1}" is true"""
                sentence2 = f"""The statement "{sentence2}" is true"""
            elif template == 2:
                sentence1 = f"""The statement "{sentence1}" is not true"""
                sentence2 = f"""The statement "{sentence2}" is not true"""

            if self.is_generative:
                sentence1 = f"Claim: {sentence1}"
                sentence2 = f"Claim: {sentence2}"

            if not self.is_generative: #self.tokenizer.pad_token is not None:
                ids_sent1 = self.tokenizer.encode(sentence1) #(f"The statement '{sentence1}' is not true") #sentence1)
            else:
                prompt = self.prompt.format(input=sentence1) #f"The statement '{sentence1}' is not true") #sentence1)
                ids_sent1 = self.tokenizer.encode(prompt)
            segs_sent1 = [0] * len(ids_sent1)

            if not self.is_generative: #self.tokenizer.pad_token is not None:
                ids_sent2 = self.tokenizer.encode(sentence2) #f"The statement '{sentence2}' is not true") #sentence2)
            else:
                prompt = self.prompt.format(input=sentence2) #f"The statement '{sentence2}' is not true") #sentence2)
                ids_sent2 = self.tokenizer.encode(prompt)
            segs_sent2 = [0] * len(ids_sent2)

            if not self.is_generative:
                if len(ids_sent1) < self.max_sent_len:
                    res = self.max_sent_len - len(ids_sent1)
                    att_mask_sent1 = [1] * len(ids_sent1) + [0] * res
                    ids_sent1 += [pad_id] * res
                    segs_sent1 += [0] * res
                else:
                    ids_sent1 = ids_sent1[:self.max_sent_len]
                    segs_sent1 = segs_sent1[:self.max_sent_len]
                    att_mask_sent1 = [1] * self.max_sent_len
                    count_truncated_samples += 1

                if len(ids_sent2) < self.max_sent_len:
                    res = self.max_sent_len - len(ids_sent2)
                    att_mask_sent2 = [1] * len(ids_sent2) + [0] * res
                    ids_sent2 += [pad_id] * res
                    segs_sent2 += [0] * res
                else:
                    ids_sent2 = ids_sent2[:self.max_sent_len]
                    segs_sent2 = segs_sent2[:self.max_sent_len]
                    att_mask_sent2 = [1] * self.max_sent_len
                    count_truncated_samples += 1
            else:
                if len(ids_sent1) < self.max_sent_len:
                    res = self.max_sent_len - len(ids_sent1)
                    att_mask_sent1 = [0] * res + [1] * len(ids_sent1) # left padding for causal lm
                    ids_sent1 = [pad_id] * res + ids_sent1
                    segs_sent1 += [0] * res
                else:
                    ids_sent1 = ids_sent1[:self.max_sent_len]
                    segs_sent1 = segs_sent1[:self.max_sent_len]
                    att_mask_sent1 = [1] * self.max_sent_len
                    count_truncated_samples += 1

                if len(ids_sent2) < self.max_sent_len:
                    res = self.max_sent_len - len(ids_sent2)
                    att_mask_sent2 = [0] * res + [1] * len(ids_sent2) # left padding for causal lm
                    ids_sent2 = [pad_id] * res + ids_sent2
                    segs_sent2 += [0] * res
                else:
                    ids_sent2 = ids_sent2[:self.max_sent_len]
                    segs_sent2 = segs_sent2[:self.max_sent_len]
                    att_mask_sent2 = [1] * self.max_sent_len
                    count_truncated_samples += 1

            if len(row) == 4:
                example = [sentence1, sentence2, ids_sent1, segs_sent1, att_mask_sent1, ids_sent2, segs_sent2, att_mask_sent2, label]
            else:
                example = [sentence1, sentence2, ids_sent1, segs_sent1, att_mask_sent1, ids_sent2, segs_sent2, att_mask_sent2]

            examples.append(example)

        print(f"finished preprocessing examples in {dataset_type}: {count_truncated_samples} samples truncated out of {len(dataset)}")

        return examples

    def read_input_files(self, file_path, name="train", return_sentences=False, template_type=0, **kwargs):
        df = pd.read_csv(file_path)
        df = df.reset_index(drop=False)
        result = df.values.tolist()

        if return_sentences and not self.config["skip_tokenizer"]:
            raise ValueError("return_sentence and skip_tokenizer are not mutually exclusive")
        if return_sentences:
            return result

        examples = self._get_examples(result, name, template=template_type)
        return examples

class OpenAIProcessor(DataProcessor):

    def __init__(self, config, num_classes):
        super(OpenAIProcessor, self).__init__(config)
        #self.highly_perturbing = config["highly_perturbing"]
        self.num_classes = num_classes

    def read_input_files(self, df, name="train", add_space=False, return_sentences=False):
        claims, evidences, labels = [], [], []

        #df = pd.read_csv(file_path)
        """if self.highly_perturbing:
            constraint = "highly_perturbing"
        else:
            constraint = "highly_unperturbing"
        """

        #df = df[df[4].str.contains(constraint)]
        sample_target = df.iloc[0,-2]
        if "support" in sample_target:
            label = [1,0] if self.num_classes == 2 else [1,0,0]
        elif "refute" in sample_target:
            label = [0,1] if self.num_classes == 2 else [0,1,0]
        elif "nei" in sample_target:
            label = [0,0,1]
        else:
            raise ValueError(f"unexpected value {sample_target}")

        for i,sample in df.iterrows():
            claims.append(sample[1])
            evidences.append(sample[2])
            labels.append(label)

        result = []
        for i, (claim, evidence, label) in enumerate(zip(claims, evidences, labels)):
            result.append([i, claim, evidence, label]) #int, string, list[string], list[int]

        if return_sentences:
            return result

        examples = self._get_examples(result, name, add_space=add_space)
        return examples

class FM2Processor(DataProcessor):

    def __init__(self, config):
        super(FM2Processor, self).__init__(config)

    def read_input_files(self, file_path, name="train", add_space=False, return_sentences=False, word=None, to_class=None, matches=[]):
        claims, evidences, labels = [], [], []

        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = json.loads(line)

                if line["label"] == "SUPPORTS":
                    label = [1,0]
                elif line["label"] == "REFUTES":
                    label = [0,1]
                else:
                    raise ValueError(f"unknown label {line['label']}")

                claims.append(line["text"])
                labels.append(label)
                evidence_text = ""
                for i, evidence in enumerate(line["gold_evidence"]):
                    if i > 0:
                        evidence_text += "\n"
                    evidence_text += evidence["text"]
                evidences.append(evidence_text)

            result = []
            for i, (claim, evidence, label) in enumerate(zip(claims, evidences, labels)):
                if word is not None:
                    to_class = to_class.lower().strip()
                    if len(matches) > 0:
                        if i not in matches:
                            continue
                        if (to_class == "support" and label[0] == 1) or (to_class == "refute" and label[1] == 1) or (len(label) == 3 and to_class == "nei" and label[2] == 1):
                            continue
                        claim, label = self.add_word(claim, label, word=word, to_class=to_class)
                result.append([i, claim, evidence, label]) #int, string, string, list[int]

        if return_sentences:
            return result

        examples = self._get_examples(result, name, add_space=add_space)
        return examples

In [10]:
def get_data(config):
  if config["dataset"] == "fm2":
      processor = FM2Processor(config)
      num_classes = 2

      path_train = "./data/fm2/train.jsonl"
      path_dev = "./data/fm2/dev.jsonl"
      path_test = "./data/fm2/test.jsonl"

  elif config["dataset"] == "antonym":
    processor = AntonymsProcessor(config)
    if "qwen" in config["model_name"].lower():
      num_classes = 3
    else:
      model_name = config["model_name"].split("/")[-1].split("_")[0]
      num_classes = 3 if any(s in model_name for s in ("avtc", "scifact", "vitaminc")) else 2

    if not config["test_only"]:
      raise ValueError("cannot train on antonyms")

    if config["stereotype"]:
      path_train = "./data/antonym/stereotype_words.csv"
      path_dev = "./data/antonym/stereotype_words.csv"
      path_test = "./data/antonym/stereotype_words.csv"
    else:
      path_train = "./data/antonym/antonym_pairs.csv"
      path_dev = "./data/antonym/antonym_pairs.csv"
      path_test = "./data/antonym/antonym_pairs.csv"
  else:
    raise ValueError(f"{config['dataset']} is not a valid dataset name (choose between 'fm2', 'antonym', 'from_openai_generated')")

  return processor, num_classes, path_train, path_dev, path_test

## Model Definitions

In [16]:
# openai

from pydantic import BaseModel, Field
from typing import Any, Dict
from openai import OpenAI
import timeout_decorator
from dotenv import load_dotenv
import re
import time

load_dotenv()

def remove_markdown_syntax(text: str) -> str:
    # remove triple backtick code blocks (```python ... ```)
    text = re.sub(r"```[\s\S]*?```", lambda m: re.sub(r"^```.*\n|```$", '', m.group()), text)

    # remove inline code (`code`)
    text = re.sub(r"`([^`]*)`", r"\1", text)

    # remove bold (**text** or __text__)
    text = re.sub(r"\*\*(.*?)\*\*", r"\1", text)

    # remove italic (*text* or _text_)
    text = re.sub(r"\*(.*?)\*", r"\1", text)

    # remove blockquotes
    text = re.sub(r"^>\s?", '', text, flags=re.MULTILINE)

    text = text.replace("python", "")
    return text.strip()

def format_prompt(prompt: str, attr: dict, **kwargs) -> str:
    return prompt.format(**attr)

def add_metadata(total_metadata, metadata):
    total_metadata["input_tokens"] += metadata['input_tokens']
    total_metadata["output_tokens"] += metadata['output_tokens']
    total_metadata["text"] += "\n" + metadata["text"]

    if "content_used" in metadata.keys():
        total_metadata["content_used"] += metadata['content_used']
        total_metadata["total_content"] += metadata['total_content']
        total_metadata["num_tables"] += 1

    return total_metadata

def extract_result(text: str, pattern: str) -> str:
    position = text.lower().rfind(pattern.lower())
    if position == -1:
        print(f"Cannot find pattern '{pattern}' in '{text}'")
        return ""
    else:
        position += len(pattern)
    return text[position:].strip()

class OpenAIModel(BaseModel):
    model_name: str = Field("gpt-4o-mini", strict=True, description="Name of the openai model as per their official website")
    temperature: float = Field(.0000000000000000000001, strict=True, description="The temperature of the model in between 0 and 1")
    top_p: float = Field(1.0, strict=True, description="The top_p of the model in between 0 and 1")
    client: Any = None # OpenAI()
    max_retries: int = Field(50, strict=True, description="Number of retries in case of failed OpenAI API call")

    def init_client(self):
        self.client = OpenAI()

    #@timeout_decorator.timeout(60, timeout_exception=StopIteration)
    def call_gpt(self, prompt: str, delay: int = 2, backoff_factor: int = 2) -> (str, dict):
        for i in range(self.max_retries):
            try:
                completion = self.client.chat.completions.create(
                        model=self.model_name,
                        messages=[
                            {
                                "role": "user",
                                "content": f"{prompt}"
                            }
                        ],
                        temperature=self.temperature,
                        top_p=self.top_p,
                        seed=42,
                )
            except:
                if i == self.max_retries - 1:
                    raise

                time.sleep(delay)
                delay *= backoff_factor
                continue
            break

        metadata = {
            "input_tokens": completion.usage.prompt_tokens,
            "output_tokens": completion.usage.completion_tokens,
        }

        return completion.choices[0].message.content, metadata


    def query(self, prompt: str, attr: dict, **kwargs) -> tuple[str, dict]:
        if self.client is None:
            self.init_client()
        if len(attr) > 0:
            prompt = format_prompt(prompt, attr)
        text = prompt

        for _ in range(self.max_retries):
            try:
                response = self.call_gpt(prompt)
                text+="\n"+response[0]
                response[1]["text"] = text
                return response
            except StopIteration:
                print("Failed to get a response. Retrying...")

        raise RuntimeError(f"Failed to query OpenAI after {self.max_retries} retries.")

openai_model = OpenAIModel()

claim_generation_prompt = """You will be given a claim and a (series of) word(s).
You must add all the provided words inside the given claim. The words added must be exactly like the one provided: any sort of stemming, lemmatization or similar is not allowed.
Also, the rest of the claim must be exactly like the original: no existing word must be removed or modified, and only the provided words must be added.
The word(s) must be added inside the claim, not at the beginning. In case of multiple words, you can add them in different positions: you are not forced to add them consecutively.
The novel claim must exclusively satisfy the following rule:
"{label}"

First reason step-by-step. Then write "Final answer: " followed exclusively by the generated claim. Do not write anything else after "Final answer: ".
Ensure that the final claim is exactly like the original, except for the added word and other stopwords.

Claim: {claim}
Word: {word}

Let's think step-by-step. """

check_claim_equivalence = """You will be given an original claim and a novel claim derived from the original.
You must determine if the following rule is satisfied or not:
"{label}"

First think step-by-step, then write "Final answer:" followed exclusively by "yes" if the rule is satisfied, "no" otherwise. Do not write anything else after "Final answer:".

Original claim: {claim1}
Novel claim: {claim2}

Let's think step-by-step."""

def generate_claim_with_openai(orig_claim, word, label_txt, label):
    claim1, _ = openai_model.query(claim_generation_prompt, attr={"claim": orig_claim, "word": word, "label": label_txt})
    if "final answer:" not in claim1.lower():
        return -1
    claim1 = extract_result(remove_markdown_syntax(claim1), "Final answer:").strip()

    for w in word.split(","):
        w = w.strip().lower()
        if w not in claim1.strip().lower():
            return -1

    check2, _ = openai_model_check.query(check_claim_equivalence, attr={"claim1": orig_claim, "claim2": claim1, "label": label_txt})
    check2 = extract_result(remove_markdown_syntax(check2), "Final answer:").strip().lower()
    if check2 != "yes":
        return -1

    return claim1

In [17]:
import torch

from transformers import AutoModel, AutoModelForCausalLM
from torch import nn
import torch.nn.functional as F

class CustomModel(nn.Module):
    def __init__(self):
        super(CustomModel, self).__init__()

    def compute_average_layers(self, embs):
        if not isinstance(embs, tuple):
            return embs
        value = None
        for emb in embs:
            if value is None:
                value = emb
            else:
                value += emb
        value /= len(embs)
        return value

class GenerativeModel(CustomModel):
    def __init__(self, config):
        super(GenerativeModel, self).__init__()

        self.num_classes = config["num_classes"]
        self.embed_size = config["embed_size"]

        # using Qwen
        self.plm = AutoModelForCausalLM.from_pretrained(
            config["model_name"],
            output_hidden_states=True,
            torch_dtype=torch.float16,
            device_map="auto"
        )

    @torch.autocast(device_type="cuda")
    def forward(self, ids_sent1, segs_sent1, att_mask_sent1):
        """
        Return full logits from Qwen’s lm_head.
        ids_sent1: [batch, seq_len]
        att_mask_sent1: [batch, seq_len]
        """
        outputs = self.plm(
            input_ids=ids_sent1,
            attention_mask=att_mask_sent1,
        )

        logits = outputs.logits[torch.arange(len(outputs.logits), device=outputs.logits.device), -1, :]

        # outputs.logits → [batch_size, seq_len, vocab_size]
        return logits

    @torch.autocast(device_type="cuda")
    def compute_concept_vector(self, ids_sent1, segs_sent1, att_mask_sent1, ids_sent2, segs_sent2, att_mask_sent2):
        # forward pass
        out_concept = self.plm(
            input_ids=ids_sent1,
            attention_mask=att_mask_sent1,
            output_hidden_states=True
        ).hidden_states[-1]  # shape: [batch, seq_len, hidden_size]

        out_random = self.plm(
            input_ids=ids_sent2,
            attention_mask=att_mask_sent2,
            output_hidden_states=True
        ).hidden_states[-1]  # shape: [batch, seq_len, hidden_size]

        out_concept = self.compute_average_layers(out_concept) # with only one layer, this is redundant
        out_random = self.compute_average_layers(out_random) # with only one layer, this is redundant

        batch_size = out_concept.size(0)
        batch_idx = torch.arange(batch_size, device=out_concept.device)

        out_concept = out_concept[batch_idx, -1, :]
        out_random = out_random[batch_idx, -1, :]

        return None, out_concept, out_random

class RobertaModel(CustomModel):
    def __init__(self, config):
        super(RobertaModel, self).__init__()

        self.num_classes = config["num_classes"]
        self.embed_size = config["embed_size"]

        if config["backbone"] is not None:
            self.plm = AutoModel.from_pretrained(config["backbone"]).to(get_device())
        else:
            self.plm = AutoModel.from_pretrained(config["model_name"]).to(get_device())

        config = self.plm.config
        config.type_vocab_size = 2
        self.plm.embeddings.token_type_embeddings = nn.Embedding(
            config.type_vocab_size, config.hidden_size
        )
        self.plm._init_weights(self.plm.embeddings.token_type_embeddings) # re-initialize token_type_embeddings to possibly accept more than 2 segment ids
        self.linear_layer = torch.nn.Linear(in_features=self.embed_size, out_features=self.num_classes)
        self._init_weights(self.linear_layer)

    def _init_weights(self, module):
        """Initialize the weights"""
        if isinstance(module, (nn.Linear, nn.Embedding)):
            module.weight.data.normal_(mean=0.0, std=self.plm.config.initializer_range)
        elif isinstance(module, nn.LayerNorm):
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)
        if isinstance(module, nn.Linear) and module.bias is not None:
            module.bias.data.zero_()

    @torch.autocast(device_type="cuda")
    def forward(self, ids_sent1, segs_sent1, att_mask_sent1):
        out_sent1 = self.plm(ids_sent1, token_type_ids=segs_sent1, attention_mask=att_mask_sent1, output_hidden_states=True)
        embed_sent1 = out_sent1.hidden_states[-1]

        H_sent = embed_sent1[:,0,:]
        predictions = self.linear_layer(H_sent)
        return predictions

    @torch.autocast(device_type="cuda")
    def compute_concept_vector(self, ids_sent1, segs_sent1, att_mask_sent1, ids_sent2, segs_sent2, att_mask_sent2):
        out_concept = self.plm(
            ids_sent1,
            token_type_ids=segs_sent1,
            attention_mask=att_mask_sent1,
            output_hidden_states=True
        )
        out_random = self.plm(
            ids_sent2,
            token_type_ids=segs_sent2,
            attention_mask=att_mask_sent2,
            output_hidden_states=True
        )
        out_concept, out_random = out_concept.hidden_states[-1], out_random.hidden_states[-1]
        out_concept, out_random = self.compute_average_layers(out_concept), self.compute_average_layers(out_random)
        out_concept = out_concept[:,0,:]
        out_random = out_random[:,0,:]

        concept_vector = out_concept - out_random
        return concept_vector, out_concept, out_random


## Trainer definition

In [15]:
import torch
import time

import numpy as np
import pandas as pd
import torch.nn as nn

from tqdm import tqdm

from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer

import torch.nn.functional as F

def max_pos(l):
    pos = 0
    max_value = -float('inf')
    for i, el in enumerate(l):
        if el > max_value:
            pos = i
            max_value = el

    return pos

class Trainer:
    def __init__(self, config, device):
        self.config = config
        self.device = device

    def train(self, epoch, model, loss_fn, optimizer, train_loader):
        epoch_start_time = time.time()
        model.train()
        tr_loss = 0

        for batch in tqdm(train_loader, desc='Iteration'):
            batch = tuple(t.to(self.device) if not isinstance(t, list) and not isinstance(t, str) else t for t in batch)
            #batch = tuple(t if not isinstance(t, list) and not isinstance(t, str) else t for t in batch)
            claim, evidence, ids_sent1, segs_sent1, att_mask_sent1, labels = batch

            out = model(ids_sent1, segs_sent1, att_mask_sent1)
            if isinstance(labels, list):
                labels = torch.tensor(np.array(labels)).to(self.device)
            loss = loss_fn(out, labels.float())

            tr_loss += loss.item()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

        timing = time.time() - epoch_start_time
        cur_lr = optimizer.param_groups[0]["lr"]
        print(f"Timing: {timing}, Epoch: {epoch + 1}, training loss: {tr_loss}, current learning rate {cur_lr}")

    def val(self, model, val_loader, return_preds=False):
        model.eval()

        loss_fn = nn.CrossEntropyLoss()

        val_loss = 0
        val_preds = []
        val_labels = []
        outs = []
        for batch in tqdm(val_loader):
            batch = tuple(t.to(self.device) if not isinstance(t, list) and not isinstance(t, str) else t for t in batch)
            claim, evidence, ids_sent1, segs_sent1, att_mask_sent1, labels = batch

            with torch.no_grad():
                out = model(ids_sent1, segs_sent1, att_mask_sent1)
                if out.shape[-1] > 3:
                    # hardcoded initial ids for "support", "refute", "not enough information" for Qwen tokenizer
                    if labels.shape[-1] != 2:
                        out.data = out.data[:, torch.tensor([1824, 83177, 537])] #refute: 83177, contrast: 12872, negate: 71219
                    else:
                        out.data = out.data[:, torch.tensor([1824, 83177])]

                preds = torch.max(out.data, 1)[1].cpu().numpy().tolist()

                loss = loss_fn(out, labels.float())
                val_loss += loss.item()

                outs.extend([out.data.cpu().numpy().tolist()[i] for i in range(len(out)) if torch.max(out.data, 1)[1].cpu().numpy().tolist()[i] == torch.max(labels,1)[1].cpu().numpy().tolist()[i]])

                labels = labels.cpu().numpy().tolist()

                val_labels.extend(labels)
                if len(labels[0]) != 2:
                    for pred in preds:
                        if pred == 0:
                            val_preds.append([1,0,0])
                        elif pred == 1:
                            val_preds.append([0,1,0])
                        else:
                            val_preds.append([0,0,1])
                else:
                    val_preds.extend([[1,0] if pred == 0 else [0,1] for pred in preds])

        print(f"val loss: {val_loss}")

        val_acc, val_prec, val_recall, val_f1 = output_metrics(val_labels, val_preds)

        if return_preds:
            return val_acc, val_prec, val_recall, val_f1, val_preds, val_labels
        return val_acc, val_prec, val_recall, val_f1

class AntonymTrainer:
    def __init__(self, config, device):
        self.config = config
        self.device = device

    def val(self, model, train_loader, **kwargs):
        concept_vectors = {}

        model.eval() # we put it to eval mode because we do not update the gradient
        with torch.no_grad():
            w = model.linear_layer.weight

            for batch in tqdm(train_loader, desc='Training...'):
                batch = tuple(t.to(self.device) if not isinstance(t, list) else t for t in batch)
                sent1, sent2, ids_sent1, segs_sent1, att_mask_sent1, ids_sent2, segs_sent2, att_mask_sent2 = batch
                out, out_sent1, out_sent2 = model.compute_concept_vector(ids_sent1, segs_sent1, att_mask_sent1, ids_sent2, segs_sent2, att_mask_sent2)

                out_normalized = out

                for i in range(len(w)):
                    w_normalized = w[i]
                    cos_sim = torch.matmul(out_normalized, w_normalized.unsqueeze(-1)).squeeze(-1)
                    for j in range(len(out)):
                        if (sent1[j], sent2[j]) not in concept_vectors.keys():
                            concept_vectors[(sent1[j], sent2[j])] = {}
                        if i == 0:
                            concept_vectors[(sent1[j], sent2[j])]["support"] = cos_sim[j].item()
                        elif i == 1:
                            concept_vectors[(sent1[j], sent2[j])]["refute"] = cos_sim[j].item()
                        else:
                            concept_vectors[(sent1[j], sent2[j])]["nei"] = cos_sim[j].item()

        return concept_vectors

    def val_potency(self, model, train_loader, num_labels=2, **kwargs):
        concept_vectors = {}

        model.eval()  # we put it to eval mode because we do not update the gradient
        with torch.no_grad():
            try:
                # roberta
                w = model.linear_layer.weight
            except:
                # qwen
                w = model.plm.lm_head.weight
                if num_labels != 2:
                    w = w[torch.tensor([1824, 83177, 537])] # for qwen2.5
                else:
                    w = w[torch.tensor([1824, 83177])] # for qwen2.5

            for batch in tqdm(train_loader, desc='Training...'):
                batch = tuple(t.to(self.device) if not isinstance(t, list) else t for t in batch)
                # words are provided in pairs following the antonym setup
                # in this case, words are not evaluated wrt each other, so we get the out_sent1 and out_sent2 to evaluate them separately

                sent1, sent2, ids_sent1, segs_sent1, att_mask_sent1, ids_sent2, segs_sent2, att_mask_sent2 = batch
                out, out_sent1, out_sent2 = model.compute_concept_vector(ids_sent1, segs_sent1, att_mask_sent1,
                                                                         ids_sent2, segs_sent2, att_mask_sent2)
                out_sent1 = out_sent1.to(w.device)
                out_sent2 = out_sent2.to(w.device)
                for i in range(len(w)):
                    cos_sim1 = torch.matmul(out_sent1, w[i].unsqueeze(-1)).squeeze(-1)
                    cos_sim2 = torch.matmul(out_sent2, w[i].unsqueeze(-1)).squeeze(-1)
                    for j in range(len(out_sent1)):
                        if sent1[j] not in concept_vectors.keys():
                            concept_vectors[sent1[j]] = {}
                        if sent2[j] not in concept_vectors.keys():
                            concept_vectors[sent2[j]] = {}

                        if i == 0:
                            concept_vectors[sent1[j]]["support"] = cos_sim1[j].item()
                            concept_vectors[sent2[j]]["support"] = cos_sim2[j].item()
                        elif i == 1:
                            concept_vectors[sent1[j]]["refute"] = cos_sim1[j].item()
                            concept_vectors[sent2[j]]["refute"] = cos_sim2[j].item()
                        else:
                            concept_vectors[sent1[j]]["nei"] = cos_sim1[j].item()
                            concept_vectors[sent2[j]]["nei"] = cos_sim2[j].item()

        return concept_vectors


## Computing Trigger Ranking

In [18]:
from torch.utils.data import DataLoader

config = {
    "model_name": "models/roberta-base/seed_1/fm2/fm2_model.pt",
    "backbone": "roberta-base",
    "dataset": "antonym",
    "embed_size": 768,
    "num_classes": 2,
    "max_sent_len": 512,
    "seed": 1,
    "batch_size": 64,
    "test_only": True,
    "stereotype": False,
    "highly_perturbing": False,
}

# we skip the handling of the NEI templates, as FM2 does not have the NEI class. See main.py for that part.
device = get_device()
set_random_seeds(config["seed"])

processor, num_classes, path_train, path_dev, path_test = get_data(config)

data_test = processor.read_input_files(path_test, name="test")

test_set = dataset(data_test)

test_dataloader = DataLoader(test_set, batch_size=config["batch_size"], shuffle=False, collate_fn=collate_fn_antonym)

if "qwen" in config["model_name"].lower():
    model = GenerativeModel(config)
else:
    model = RobertaModel(config)

model.to(device)

if config["model_name"].split(".")[-1] == "pt" and config["backbone"] is not None:
    # Load the finetuned model
    state_dict = torch.load(config["model_name"], map_location=device)
    model.load_state_dict(state_dict)

    model.to(device)

trainer = AntonymTrainer(config, device)

# ranking triggers

concept_vectors = trainer.val_potency(model, test_dataloader, num_labels=num_classes)

list_concepts = [[]]
concepts = [concept_vectors]
for i in range(len(list_concepts)):
    for k,v in concepts[i].items():
        tmp = []
        tmp.append(k)
        for k1,v1 in v.items():
            tmp.append(v1)
        list_concepts[i].append(tmp)

list_concept_vectors = list_concepts[0]

columns = ["Word pair", "Support", "Refute"]

ranked_df = pd.DataFrame(list_concept_vectors, columns=columns) # word rankings

tokenizing...: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3556/3556 [00:00<00:00, 11182.42it/s]


finished preprocessing examples in test: 0 samples truncated out of 3556


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Training...: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 56/56 [00:12<00:00,  4.33it/s]


## Extracting the dataset instances correctly predicted

In [19]:
def defend(dataloader):
    """
    evaluation function
    """
    val_labels, val_preds, claims, evidences = [], [], [], []
    for batch in dataloader:
        batch = tuple(t.to(get_device()) if not isinstance(t, list) and not isinstance(t, str) else t for t in batch)
        claim, evidence, ids_sent1, segs_sent1, att_mask_sent1, labels = batch

        with torch.no_grad():
            out = model(ids_sent1, segs_sent1, att_mask_sent1)
            if out.shape[-1] > 3: # generative model
                if labels.shape[-1] != 2:
                    # hardcoded initial ids for "support", "refute", "not enough information" for Qwen tokenizer
                    out.data = out.data[:, torch.tensor([1824, 83177, 537])]
                else:
                    # hardcoded initial ids for "support" and "refute" for Qwen tokenizer
                    out.data = out.data[:, torch.tensor([1824, 83177])]

            preds = torch.max(out.data, 1)[1].cpu().numpy().tolist()
            labels_pos = torch.max(labels, 1)[1].cpu().numpy().tolist()
            val_labels.extend(labels_pos)
            val_preds.extend(preds)

        claims.extend(claim)
        evidences.extend(evidence)

    return val_labels, val_preds

def get_matching_samples(predictions, targets):
    """
    Return the list of indices where predictions match targets.
    """
    assert len(predictions) == len(targets)
    matching_samples = []
    for i in range(len(predictions)):
        if predictions[i] == targets[i]:
            matching_samples.append(i)

    return matching_samples

def get_samples_by_position(data, positions):
    """
    Return the samples from data at the specified positions.
    """
    new_data = [data[position] for position in positions]
    return new_data

config["dataset"] = "fm2"
processor, num_classes, path_train, path_dev, path_test = get_data(config)

is_generative = "qwen" in config["model_name"].lower()

data_test = processor.read_input_files(path_test, name="test")
if is_generative:
    data_test = data_test[:500] # reducing the size for faster evaluation for Qwen

test_set = dataset(data_test)
test_dataloader = DataLoader(test_set, batch_size=config["batch_size"], shuffle=False, collate_fn=collate_fn)

predictions_test, targets_test = defend(test_dataloader)

test_match = get_matching_samples(predictions_test, targets_test)

data_test = processor.read_input_files(path_test, name="test")
test = get_samples_by_position(data_test, test_match)
test = pd.DataFrame(test).iloc[:500]

tokenizing...: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1380/1380 [00:00<00:00, 2602.72it/s]


finished preprocessing examples in test: 0 samples truncated out of 1380


tokenizing...: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1380/1380 [00:00<00:00, 2817.68it/s]

finished preprocessing examples in test: 0 samples truncated out of 1380


## Integration of the words inside the claims

In this example, we will create the datasets using FactFlip-RAW (raw trigger injection) for high and low perturbing triggers. In this way, we can evaluate the impact of trigger words on the model's predictions, and evaluate the trigger discrimination of FactFlip (i.e. highly perturbing triggers should have a higher impact on the model's predictions compared to low perturbing triggers).

The FactFlip-RAW setting, while it does not include the perturb-and-verify pipeline, is still indicative of the attack strength and the discrimination power of the triggers, as shown in the FactFlip paper.

In [20]:
from copy import deepcopy

def get_testing_concepts(concept_vectors, k=5):
    """
    For both high and low perturbing cases, we return a dictionary containing the top-k and bottom-k triggers.
    """

    if k == -1:
        # used when evaluating stereotype words
        # this output format is to be consistent with the general pipeline for evaluating adversarial words
        return {
            "highly_perturbing": concept_vectors.values.tolist(),
            "highly_unperturbing": [], #concept_vectors.values.tolist()
        }
    concept_vectors_list = concept_vectors.values.tolist()
    highly_perturbing = concept_vectors_list[:k]
    highly_unperturbing = concept_vectors_list[-k:]

    return {
        "highly_perturbing": highly_perturbing,
        "highly_unperturbing": highly_unperturbing
    }

def cv_attack(test, sampled_concepts, to="support", from_template=True):
    if from_template:
        return attack_from_template(test, sampled_concepts, to)
    else:
        return attack(test, sampled_concepts, to)

# FactFlip-RAW attack
def attack_from_template(test, sampled_concepts, to="support"):
    test_samples = test
    perturbing_samples = [[] for _ in range(len(sampled_concepts))]

    for i, perturbation_type in enumerate(tqdm(sampled_concepts.keys())):
        for j, concept in enumerate(sampled_concepts[perturbation_type]):
            for sample in test_samples:
                word = concept[0]
                claim = f"{word}. {sample[0]}"
                perturbing_samples[i].append([len(perturbing_samples[i]) - 1, claim, sample[1], sample[-1], f"{perturbation_type}_{to}"])

    df = pd.DataFrame([row + [i] for i, sublist in enumerate(perturbing_samples) for row in sublist])
    return df

# FactFlip
def attack(test, sampled_concepts, to="support"):
    test_samples = test
    perturbing_samples = [[] for _ in range(len(sampled_concepts))]

    for i, perturbation_type in enumerate(tqdm(sampled_concepts.keys())):
        for j, concept in enumerate(sampled_concepts[perturbation_type]):
            for sample in tqdm(test_samples):
                if sample[-1][0] == 1:
                    label_txt = "the original claim must logically entail the new claim"
                elif sample[-1][1] == 1:
                    label_txt = "the new claim must logically entail the original claim"
                else:
                    label_txt = "the original claim must logically entail the new claim and vice versa"

                word = concept[0]
                claim1 = generate_claim_with_openai(sample[0], word, label_txt, sample[-1])
                if claim1 == -1:
                    continue

                perturbing_samples[i].append([len(perturbing_samples[i])-1, claim1, sample[1], sample[-1], f"{perturbation_type}_{to}"])

    df = pd.DataFrame([row + [i] for i, sublist in enumerate(perturbing_samples) for row in sublist])
    return df

# trying to flip to support prediction
# we take non-supporting predictions and we analyze which perturbations modify the predictions the most

k = 5 # number of triggers to use for each perturbation type
test_samples = test[test.iloc[:, -1].apply(lambda x: x[0] == 0)]
test_samples = deepcopy(test_samples.values.tolist())

# support
print("Generating support...")
concept_vectors = ranked_df.sort_values(by='Support', ascending=False)
testing_concepts = get_testing_concepts(concept_vectors, k)

df_support = cv_attack(test_samples, testing_concepts, to="support", from_template=True) #change to false to run the perturb-and-verify pipeline

# we do the same for refute
test_samples = test[test.iloc[:, -1].apply(lambda x: x[1] == 0)]
test_samples = deepcopy(test_samples.values.tolist())

print("Generating refute...")
concept_vectors = ranked_df.sort_values(by='Refute', ascending=False)
testing_concepts = get_testing_concepts(concept_vectors, k)

df_refute = cv_attack(test_samples, testing_concepts, to="refute", from_template=True) #change to false to run the perturb-and-verify pipeline

Generating support...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 2045.50it/s]


Generating refute...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 2146.52it/s]


## Testing

In [21]:
num_classes = 2
processor = OpenAIProcessor(config, num_classes)

trainer = Trainer(config, device)

df_support_pert = df_support[df_support[4] == "highly_perturbing_support"]
df_refute_pert = df_refute[df_refute[4] == "highly_perturbing_refute"]
df_support_unpert = df_support[df_support[4] == "highly_unperturbing_support"]
df_refute_unpert = df_refute[df_refute[4] == "highly_unperturbing_refute"]

data_test_support_pert = processor.read_input_files(df_support_pert, name="test")
data_test_refute_pert = processor.read_input_files(df_refute_pert, name="test")
data_test_support_unpert = processor.read_input_files(df_support_unpert, name="test")
data_test_refute_unpert = processor.read_input_files(df_refute_unpert, name="test")

if "qwen" in config["model_name"].lower(): # we reduce the amount of samples for qwen
    data_test = data_test[:500]

test_set_support_pert = dataset(data_test_support_pert)
test_set_refute_pert = dataset(data_test_refute_pert)
test_set_support_unpert = dataset(data_test_support_unpert)
test_set_refute_unpert = dataset(data_test_refute_unpert)

test_support_pert_dataloader = DataLoader(test_set_support_pert, batch_size=config["batch_size"], shuffle=False, collate_fn=collate_fn)
test_refute_pert_dataloader = DataLoader(test_set_refute_pert, batch_size=config["batch_size"], shuffle=False, collate_fn=collate_fn)
test_support_unpert_dataloader = DataLoader(test_set_support_unpert, batch_size=config["batch_size"], shuffle=False, collate_fn=collate_fn)
test_refute_unpert_dataloader = DataLoader(test_set_refute_unpert, batch_size=config["batch_size"], shuffle=False, collate_fn=collate_fn)

test_a_sup_pert, test_p_sup_pert, test_r_sup_pert, test_f1_sup_pert, predictions_sup_pert, labels_sup_pert = trainer.val(model, test_support_pert_dataloader, return_preds=True)
test_a_ref_pert, test_p_ref_pert, test_r_ref_pert, test_f1_ref_pert, predictions_ref_pert, labels_ref_pert = trainer.val(model, test_refute_pert_dataloader, return_preds=True)
test_a_sup_unpert, test_p_sup_unpert, test_r_sup_unpert, test_f1_sup_unpert, predictions_sup_unpert, labels_sup_unpert = trainer.val(model, test_support_unpert_dataloader, return_preds=True)
test_a_ref_unpert, test_p_ref_unpert, test_r_ref_unpert, test_f1_ref_unpert, predictions_ref_unpert, labels_ref_unpert = trainer.val(model, test_refute_unpert_dataloader, return_preds=True)

print("Perturbing - support attack:")
print(f"Accuracy: {test_a_sup_pert}, Precision: {test_p_sup_pert}, Recall: {test_r_sup_pert}, F1: {test_f1_sup_pert}")
print("Perturbing - refute attack:")
print(f"Accuracy: {test_a_ref_pert}, Precision: {test_p_ref_pert}, Recall: {test_r_ref_pert}, F1: {test_f1_ref_pert}")
print("Unperturbing - support attack:")
print(f"Accuracy: {test_a_sup_unpert}, Precision: {test_p_sup_unpert}, Recall: {test_r_sup_unpert}, F1: {test_f1_sup_unpert}")
print("Unperturbing - refute attack:")
print(f"Accuracy: {test_a_ref_unpert}, Precision: {test_p_ref_unpert}, Recall: {test_r_ref_unpert}, F1: {test_f1_ref_unpert}")

tokenizing...: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1270/1270 [00:00<00:00, 2766.14it/s]


finished preprocessing examples in test: 0 samples truncated out of 1270


tokenizing...: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1230/1230 [00:00<00:00, 2659.75it/s]


finished preprocessing examples in test: 0 samples truncated out of 1230


tokenizing...: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1270/1270 [00:00<00:00, 1551.68it/s]


...

val loss: 141.34027528762817
accuracy:      0.125203
precision:     0.500000
recall:        0.062602
f1:            0.111272


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:02<00:00,  8.52it/s]


val loss: 198.65463256835938
accuracy:      0.016535
precision:     0.500000
recall:        0.008268
f1:            0.016266


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:02<00:00,  8.82it/s]

val loss: 170.01009273529053
accuracy:      0.018699
precision:     0.500000
recall:        0.009350
f1:            0.018356
Perturbing - support attack:
Accuracy: 0.0984251968503937, Precision: 0.5, Recall: 0.04921259842519685, F1: 0.08960573476702509
Perturbing - refute attack:
Accuracy: 0.12520325203252033, Precision: 0.5, Recall: 0.06260162601626017, F1: 0.11127167630057803
Unperturbing - support attack:
Accuracy: 0.01653543307086614, Precision: 0.5, Recall: 0.00826771653543307, F1: 0.016266460108443067
Unperturbing - refute attack:
Accuracy: 0.01869918699186992, Precision: 0.5, Recall: 0.00934959349593496, F1: 0.018355945730247406


The results are the same as the ones shown in the paper in Table 8. At the same time, we can see how much the trigger ranking method is discriminative: for both the *support* and *refute* classes, injecting the perturbing words greatly increases the attack success rate wrt the unperturbing words.